# Dynamic Lakehouse Shortcut Creation

This notebook creates shortcuts from a source lakehouse to a schema-enabled destination lakehouse.

**Pattern:**
- Source: Tables with a specific prefix (e.g., `adp__account`, `adp__employee`) OR tables in a schema
- Destination: Schema-enabled lakehouse where tables are created without prefix (e.g., `adp.account`, `adp.employee`)

**Supports:**
- Non-schema source → Schema destination (prefix-based migration)
- Schema source → Schema destination (cross-schema shortcuts)

**Prerequisites:**
- Semantic Link Labs installed
- Source and destination lakehouses exist
- Appropriate permissions on both workspaces
- **Run as Python notebook for optimal performance**

## 1. Install/Upgrade Semantic Link Labs

Ensure you have version 0.12.9 or later for schema support

In [58]:
# Install/upgrade semantic-link-labs
# %pip install --upgrade semantic-link-labs

## 2. Configuration

Set your source and destination parameters

In [59]:
# Source lakehouse configuration
SOURCE_WORKSPACE = "Staging"              # Source workspace name
SOURCE_LAKEHOUSE = "landing_lh__fivetran"   # Source lakehouse name
SOURCE_SCHEMA = None                       # Set to schema name if source is schema-enabled, None for non-schema
TABLE_PREFIX = "samsara__"                     # For non-schema: prefix to filter (e.g., 'adp__'). For schema: optional filter within schema

# Destination lakehouse configuration  
DEST_WORKSPACE = "Data Engineering [Dev]"  # Destination workspace name
DEST_LAKEHOUSE = "Bronze_Lake"             # Destination lakehouse name (must be schema-enabled)
DEST_SCHEMA = "Samsara"                        # Schema name in destination lakehouse

# Shortcut behavior
CONFLICT_POLICY = "Abort"                  # Options: 'Abort' or 'CreateOrOverwrite'
DRY_RUN = True                             # Set to True to preview without creating shortcuts

# Optional: Manually specify tables if you don't want to auto-discover
# Leave as None to auto-discover based on prefix
MANUAL_TABLE_LIST = None  # Example: ['adp__account', 'adp__employee', 'adp__payroll']

## 3. Import Libraries

In [60]:
import sempy_labs as labs
import sempy_labs.lakehouse as lake
import pandas as pd
from typing import List, Dict
import re

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 4. Helper Functions

In [61]:
def get_tables_by_prefix(
    lakehouse: str, 
    workspace: str, 
    schema: str = None, 
    prefix: str = ""
) -> List[Dict[str, str]]:
    """
    Get tables from source lakehouse using Semantic Link Labs REST API.
    Supports both schema-enabled and non-schema-enabled lakehouses.
    
    Args:
        lakehouse: Source lakehouse name
        workspace: Source workspace name
        schema: Schema name (for schema-enabled lakehouses), None for non-schema
        prefix: Table name prefix/pattern to filter
    
    Returns:
        List of dicts with 'source_table', 'source_schema', and 'source_path' keys
    """
    try:
        print(f"📊 Discovering tables in {workspace}/{lakehouse}")
        if schema:
            print(f"   Schema: {schema}")
        if prefix:
            print(f"   Prefix filter: {prefix}")
        
        # Get all tables from lakehouse using Semantic Link Labs
        tables_df = lake.get_lakehouse_tables(
            lakehouse=lakehouse,
            workspace=workspace
        )
        
        if tables_df.empty:
            print("   No tables found in lakehouse")
            return []
        
        results = []
        prefix_clean = prefix.rstrip('%') if prefix else ""
        
        for _, row in tables_df.iterrows():
            # Extract table info from separate columns
            table_name = row['Table Name']
            table_schema = row.get('Schema Name', None)  # May be None for non-schema tables
            table_type = row.get('Type', 'Unknown')
            
            # Apply schema filter if specified
            if schema and table_schema != schema:
                # Skip tables not in target schema
                continue
            
            # Apply prefix filter if specified
            if prefix_clean and not table_name.startswith(prefix_clean):
                continue
            
            # Determine source path based on whether table is in a schema
            if table_schema:
                # Schema-enabled source
                source_path = f"Tables/{table_schema}"
            else:
                # Non-schema source
                source_path = "Tables"
            
            results.append({
                'source_table': table_name,
                'source_schema': table_schema,
                'source_path': source_path,
                'table_type': table_type
            })
        
        print(f"✓ Found {len(results)} table(s) matching criteria\n")
        return results
        
    except Exception as e:
        print(f"✗ Error discovering tables: {e}")
        import traceback
        traceback.print_exc()
        return []

In [62]:
def create_shortcuts_batch(
    tables: List[Dict[str, str]],
    source_lakehouse: str,
    source_workspace: str,
    dest_lakehouse: str,
    dest_workspace: str,
    dest_schema: str,
    conflict_policy: str = "Abort",
    dry_run: bool = True
) -> Dict[str, List[Dict]]:
    """
    Create shortcuts for multiple tables with schema organization.
    
    Args:
        tables: List of dicts with 'source_table', 'source_schema', 'source_path'
        source_lakehouse: Source lakehouse name
        source_workspace: Source workspace name
        dest_lakehouse: Destination lakehouse name
        dest_workspace: Destination workspace name
        dest_schema: Destination schema name
        conflict_policy: 'Abort' or 'CreateOrOverwrite'
        dry_run: If True, only preview without creating
    
    Returns:
        Dict with 'success', 'failed', and 'skipped' lists
    """
    results = {'success': [], 'failed': [], 'skipped': []}
    
    print(f"{'='*80}")
    print(f"📋 BATCH SHORTCUT CREATION")
    print(f"{'='*80}")
    print(f"Source: {source_workspace}/{source_lakehouse}")
    print(f"Destination: {dest_workspace}/{dest_lakehouse}.{dest_schema}")
    print(f"Mode: {'DRY RUN' if dry_run else 'LIVE'}")
    print(f"Conflict Policy: {conflict_policy}")
    print(f"{'='*80}\n")
    
    for i, table_info in enumerate(tables, 1):
        source_table = table_info['source_table']
        source_path = table_info['source_path']
        
        # Remove prefix for destination table name
        prefix_clean = TABLE_PREFIX.rstrip('%') if TABLE_PREFIX else ""
        if prefix_clean and source_table.startswith(prefix_clean):
            dest_table_name = source_table[len(prefix_clean):]
        else:
            dest_table_name = source_table
        
        print(f"[{i}/{len(tables)}] Processing: {source_table}")
        print(f"  Source path: {source_path}/{source_table}")
        print(f"  Destination: {dest_schema}.{dest_table_name}")
        
        if dry_run:
            print(f"  ✓ Would create shortcut (DRY RUN)\n")
            results['success'].append({
                'source_table': source_table,
                'dest_table': dest_table_name,
                'error': None
            })
            continue
        
        try:
            # Create the shortcut using latest version signature
            lake.create_shortcut_onelake(
                table_name=source_table,
                source_workspace=source_workspace,
                source_item=source_lakehouse,
                source_item_type='Lakehouse',
                source_path=source_path,
                destination_lakehouse=dest_lakehouse,
                destination_workspace=dest_workspace,
                destination_path=f'Tables/{dest_schema}',
                shortcut_name=dest_table_name,
                shortcut_conflict_policy=conflict_policy
            )
            
            print(f"  ✓ Shortcut created successfully\n")
            results['success'].append({
                'source_table': source_table,
                'dest_table': dest_table_name,
                'error': None
            })
            
        except Exception as e:
            error_str = str(e)
            print(f"  ✗ Error: {error_str}\n")
            
            # Check if it's a "already exists" error
            if 'already exists' in error_str.lower() or 'conflict' in error_str.lower():
                results['skipped'].append({
                    'source_table': source_table,
                    'dest_table': dest_table_name,
                    'reason': 'Already exists'
                })
            else:
                results['failed'].append({
                    'source_table': source_table,
                    'dest_table': dest_table_name,
                    'error': error_str
                })
    
    return results

## 5. Discover Tables

Find all tables matching the specified criteria

In [63]:
# Discover or use manual table list
if MANUAL_TABLE_LIST:
    print("📝 Using manually specified table list\n")
    # For manual list, construct table info based on SOURCE_SCHEMA setting
    if SOURCE_SCHEMA:
        tables = [{
            'source_table': t,
            'source_schema': SOURCE_SCHEMA,
            'source_path': f'Tables/{SOURCE_SCHEMA}',
            'table_type': 'Manual'
        } for t in MANUAL_TABLE_LIST]
    else:
        tables = [{
            'source_table': t,
            'source_schema': None,
            'source_path': 'Tables',
            'table_type': 'Manual'
        } for t in MANUAL_TABLE_LIST]
else:
    print("🔍 Auto-discovering tables...\n")
    tables = get_tables_by_prefix(
        lakehouse=SOURCE_LAKEHOUSE,
        workspace=SOURCE_WORKSPACE,
        schema=SOURCE_SCHEMA,
        prefix=TABLE_PREFIX
    )

if not tables:
    print("⚠️  No tables found matching criteria")
else:
    print(f"\n{'='*80}")
    print(f"📊 DISCOVERED TABLES")
    print(f"{'='*80}")
    
    # Preview what will be created
    preview_data = []
    prefix_clean = TABLE_PREFIX.rstrip('%') if TABLE_PREFIX else ""
    
    for table in tables:
        source_table = table['source_table']
        
        # Calculate destination name
        if prefix_clean and source_table.startswith(prefix_clean):
            dest_table = source_table[len(prefix_clean):]
        else:
            dest_table = source_table
        
        preview_data.append({
            'Source_Table': source_table,
            'Source_Path': table['source_path'],
            'Dest_Schema': DEST_SCHEMA,
            'Dest_Table': dest_table
        })
    
    preview_df = pd.DataFrame(preview_data)
    display(preview_df)
    
    print(f"\n✓ Found {len(tables)} table(s) to process")
    print(f"\n💡 Review the table above. Set DRY_RUN = False in Cell 2 to create shortcuts.")

🔍 Auto-discovering tables...

📊 Discovering tables in Staging/landing_lh__fivetran
   Prefix filter: samsara__
✓ Found 35 table(s) matching criteria


📊 DISCOVERED TABLES



✓ Found 35 table(s) to process

💡 Review the table above. Set DRY_RUN = False in Cell 2 to create shortcuts.


## 6. Create Shortcuts

Execute shortcut creation based on discovered tables

In [64]:
if not tables:
    print("❌ No tables to process. Check your configuration and re-run.")
else:
    # Create shortcuts
    results = create_shortcuts_batch(
        tables=tables,
        source_lakehouse=SOURCE_LAKEHOUSE,
        source_workspace=SOURCE_WORKSPACE,
        dest_lakehouse=DEST_LAKEHOUSE,
        dest_workspace=DEST_WORKSPACE,
        dest_schema=DEST_SCHEMA,
        conflict_policy=CONFLICT_POLICY,
        dry_run=DRY_RUN
    )
    
    # Summary
    print(f"\n{'='*80}")
    print(f"📊 SUMMARY")
    print(f"{'='*80}")
    print(f"✓ Success: {len(results['success'])}")
    print(f"✗ Failed: {len(results['failed'])}")
    print(f"⊘ Skipped: {len(results['skipped'])}")
    print(f"{'='*80}\n")
    
    if results['failed']:
        print("\n❌ FAILED SHORTCUTS:")
        for item in results['failed']:
            print(f"  - {item['source_table']}: {item['error']}")
    
    if results['skipped']:
        print("\n⊘ SKIPPED SHORTCUTS (already exist):")
        for item in results['skipped']:
            print(f"  - {item['source_table']} → {DEST_SCHEMA}.{item['dest_table']}")

📋 BATCH SHORTCUT CREATION
Source: Staging/landing_lh__fivetran
Destination: Data Engineering [Dev]/Bronze_Lake.Samsara
Mode: LIVE
Conflict Policy: Abort

[1/35] Processing: samsara__vehicle_idling_report
  Source path: Tables/samsara__vehicle_idling_report
  Destination: Samsara.vehicle_idling_report
🟢 The shortcut 'vehicle_idling_report' was created in the 'Bronze_Lake' lakehouse within the 'Data Engineering [Dev]' workspace. It is based on the 'samsara__vehicle_idling_report' table in the 'landing_lh__fivetran' Lakehouse within the 'Staging' workspace.
  ✓ Shortcut created successfully

[2/35] Processing: samsara__vehicle
  Source path: Tables/samsara__vehicle
  Destination: Samsara.vehicle
🟢 The shortcut 'vehicle' was created in the 'Bronze_Lake' lakehouse within the 'Data Engineering [Dev]' workspace. It is based on the 'samsara__vehicle' table in the 'landing_lh__fivetran' Lakehouse within the 'Staging' workspace.
  ✓ Shortcut created successfully

[3/35] Processing: samsara__equi

## 7. Results Table

Detailed results of the operation

In [65]:
if tables:
    # Create detailed results dataframe
    all_results = []
    
    for item in results['success']:
        all_results.append({
            'Source_Table': item['source_table'],
            'Destination_Schema': DEST_SCHEMA,
            'Destination_Table': item['dest_table'],
            'Status': 'Success',
            'Error': None
        })
    
    for item in results['failed']:
        all_results.append({
            'Source_Table': item['source_table'],
            'Destination_Schema': DEST_SCHEMA,
            'Destination_Table': item['dest_table'],
            'Status': 'Failed',
            'Error': item['error']
        })
    
    for item in results['skipped']:
        all_results.append({
            'Source_Table': item['source_table'],
            'Destination_Schema': DEST_SCHEMA,
            'Destination_Table': item['dest_table'],
            'Status': 'Skipped',
            'Error': item['reason']
        })
    
    results_df = pd.DataFrame(all_results)
    display(results_df)

## Notes and Best Practices

**Concept: Why remove prefixes and use schemas?**
- Source systems often use prefixes to organize tables (e.g., `adp__`, `salesforce__`, `netsuite__`)
- Schema-enabled lakehouses provide a better organizational structure
- Moving from `adp__account` → `adp.account` provides cleaner naming and better lineage

**Pattern Explanation:**
- `SOURCE_SCHEMA = None`: Source is non-schema lakehouse (tables have prefixes like `adp__account`)
- `SOURCE_SCHEMA = "raw"`: Source is schema-enabled lakehouse (tables are like `raw.account`)
- `TABLE_PREFIX = "adp__"`: Filter pattern to match tables
- Prefix gets removed: `adp__account` becomes `account`
- Shortcut created at: `Tables/{DEST_SCHEMA}/{dest_table_name}`
- Resulting path: `adp.account` in the schema-enabled lakehouse

**Common Patterns:**
1. **Non-schema → Schema (Prefix Migration)**
   ```python
   SOURCE_SCHEMA = None
   TABLE_PREFIX = "adp__"
   DEST_SCHEMA = "adp"
   # adp__account → adp.account
   ```

2. **Schema → Schema (Cross-schema shortcuts)**
   ```python
   SOURCE_SCHEMA = "raw"
   TABLE_PREFIX = ""
   DEST_SCHEMA = "bronze"
   # raw.customers → bronze.customers
   ```

3. **Schema → Schema with filter**
   ```python
   SOURCE_SCHEMA = "salesforce"
   TABLE_PREFIX = "Account"
   DEST_SCHEMA = "crm"
   # salesforce.Account → crm.Account
   # salesforce.AccountHistory → crm.AccountHistory
   ```

**Why Python notebook instead of PySpark?**
- Shortcut creation uses Fabric REST APIs (no Spark needed)
- Python notebooks start ~10x faster than PySpark
- Uses `labs.list_tables()` REST API instead of `spark.sql()`

**Troubleshooting:**
- If shortcuts don't appear: Ensure destination lakehouse is schema-enabled
- Name validation errors: Check for special characters in table names
- Permission errors: Verify contributor access on both workspaces
- Empty table list: Check SOURCE_SCHEMA and TABLE_PREFIX settings
- "Schema doesn't exist": Create schema first via Fabric UI or notebook
